In [3]:
!pip install ipywidgets

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   ---------------------------------------- 0/3 [widgetsnbextension]
   ---------------------------------------- 0/3 [widgetsnbextension]
   ---------------------------------------- 0/3 [widgetsnbextension]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets

In [2]:
from sentence_transformers import SentenceTransformer
from typing import List
import torch

class QwenEmbeddingModel:
    """
    Wrapper for Qwen3-Embedding-4B model using sentence-transformers
    """
    
    def __init__(self, model_name: str = "Qwen/Qwen3-Embedding-4B"):
        """
        Initialize Qwen embedding model
        
        Args:
            model_name: HuggingFace model identifier
        """
        # Check if GPU is available
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        
        # Load model
        print(f"Loading {model_name}...")
        self.model = SentenceTransformer(model_name)
        self.model.to(self.device)
        self.embedding_dim = 2048  # Qwen3-Embedding-4B output dimension
        
        print(f"Model loaded. Embedding dimension: {self.embedding_dim}")
    
    def embed_texts(self, texts: List[str], batch_size: int = 32) -> List[List[float]]:
        """
        Embed a list of texts
        
        Args:
            texts: List of text strings to embed
            batch_size: Batch size for processing (adjust based on GPU memory)
        
        Returns:
            List of embedding vectors
        """
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        return embeddings.tolist()
    
    def embed_single(self, text: str) -> List[float]:
        """
        Embed a single text
        
        Args:
            text: Single text string to embed
        
        Returns:
            Embedding vector
        """
        embedding = self.model.encode(text, convert_to_numpy=True)
        return embedding.tolist()
    
    def embed_query(self, query: str) -> List[float]:
        """
        Embed a query (same as embed_single, but for clarity)
        
        Args:
            query: Query string to embed
        
        Returns:
            Embedding vector
        """
        return self.embed_single(query)

# Test the embedding model
if __name__ == "__main__":
    # Initialize
    embedding_model = QwenEmbeddingModel()
    
    # Test with sample texts
    test_texts = [
        "Personal information includes names and addresses.",
        "Organizations must collect data with consent.",
        "The weather is nice today."
    ]
    
    print("\n=== Testing Embedding Model ===")
    embeddings = embedding_model.embed_texts(test_texts)
    
    print(f"Input texts: {len(test_texts)}")
    print(f"Embeddings shape: {len(embeddings)}x{len(embeddings[0])}")
    
    # Calculate similarity between first two texts
    import numpy as np
    
    def cosine_similarity(a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
    
    sim_01 = cosine_similarity(embeddings[0], embeddings[1])
    sim_02 = cosine_similarity(embeddings[0], embeddings[2])
    
    print(f"\nSimilarity between text 0 and 1: {sim_01:.4f} (similar topics)")
    print(f"Similarity between text 0 and 2: {sim_02:.4f} (different topics)")

d:\conda_envs\aiapf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Loading Qwen/Qwen3-Embedding-4B...


d:\conda_envs\aiapf\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ianwa\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-4B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]Xet Storage is enabled for this repo, but the 'hf_xet' package is not 

Model loaded. Embedding dimension: 2048

=== Testing Embedding Model ===


Batches: 100%|██████████| 1/1 [00:03<00:00,  3.32s/it]

Input texts: 3
Embeddings shape: 3x2560

Similarity between text 0 and 1: 0.4940 (similar topics)
Similarity between text 0 and 2: 0.3572 (different topics)


In [4]:
import pdfplumber
from typing import List, Dict, Tuple
import os

class PDFDocumentExtractor:
    """
    Extract text from PDF documents using pdfplumber
    """
    
    def __init__(self, pdf_path: str):
        """
        Initialize PDF extractor
        
        Args:
            pdf_path: Path to PDF file
        """
        if not os.path.exists(pdf_path):
            raise FileNotFoundError(f"PDF not found: {pdf_path}")
        
        self.pdf_path = pdf_path
        self.pdf = pdfplumber.open(pdf_path)
        self.metadata = self._extract_metadata()
    
    def _extract_metadata(self) -> Dict:
        """Extract PDF metadata"""
        metadata = {
            "filename": os.path.basename(self.pdf_path),
            "filepath": self.pdf_path,
            "total_pages": len(self.pdf.pages),
            "pdf_metadata": self.pdf.metadata if self.pdf.metadata else {}
        }
        return metadata
    
    def extract_full_text(self) -> str:
        """
        Extract all text from PDF
        
        Returns:
            Full text content
        """
        full_text = ""
        
        for page_num, page in enumerate(self.pdf.pages, 1):
            text = page.extract_text()
            if text:
                # Add page marker for tracking
                full_text += f"\n[PAGE {page_num}]\n"
                full_text += text
        
        return full_text
    
    def extract_text_with_pages(self) -> List[Dict]:
        """
        Extract text from each page separately
        
        Returns:
            List of dicts with page content and metadata
        """
        pages_data = []
        
        for page_num, page in enumerate(self.pdf.pages, 1):
            text = page.extract_text()
            
            if text:
                pages_data.append({
                    "page_number": page_num,
                    "text": text,
                    "text_length": len(text),
                    "char_count": len(text),
                    "has_content": True
                })
            else:
                pages_data.append({
                    "page_number": page_num,
                    "text": "",
                    "text_length": 0,
                    "char_count": 0,
                    "has_content": False
                })
        
        return pages_data
    
    def extract_tables(self) -> List[Dict]:
        """
        Extract tables from PDF (if any)
        
        Returns:
            List of extracted tables with page info
        """
        tables = []
        
        for page_num, page in enumerate(self.pdf.pages, 1):
            page_tables = page.extract_tables()
            
            if page_tables:
                for table_idx, table in enumerate(page_tables):
                    tables.append({
                        "page_number": page_num,
                        "table_index": table_idx,
                        "rows": len(table),
                        "columns": len(table[0]) if table else 0,
                        "table_data": table
                    })
        
        return tables
    
    def get_page_text(self, page_number: int) -> str:
        """
        Get text from specific page
        
        Args:
            page_number: Page number (1-indexed)
        
        Returns:
            Text content of page
        """
        if page_number < 1 or page_number > len(self.pdf.pages):
            raise ValueError(f"Invalid page number: {page_number}")
        
        page = self.pdf.pages[page_number - 1]
        return page.extract_text()
    
    def get_statistics(self) -> Dict:
        """Get PDF statistics"""
        pages_data = self.extract_text_with_pages()
        
        total_chars = sum(p["char_count"] for p in pages_data)
        pages_with_content = sum(1 for p in pages_data if p["has_content"])
        
        return {
            "total_pages": self.metadata["total_pages"],
            "pages_with_content": pages_with_content,
            "total_characters": total_chars,
            "estimated_tokens": total_chars // 4,  # Rough estimate
            "average_chars_per_page": total_chars // pages_with_content if pages_with_content > 0 else 0
        }
    
    def close(self):
        """Close PDF file"""
        self.pdf.close()
    
    def __enter__(self):
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.close()

# Test PDF extraction
if __name__ == "__main__":
    pdf_path = r"D:\PersonalProjs\2SAF\US_TRADOC\test_data\ARN44767-FM_3-01-000-WEB-1.pdf"  
    
    print("=== Testing PDF Extraction ===")
    
    with PDFDocumentExtractor(pdf_path) as extractor:
        # Print metadata
        print(f"PDF: {extractor.metadata['filename']}")
        print(f"Pages: {extractor.metadata['total_pages']}")
        
        # Get statistics
        stats = extractor.get_statistics()
        print(f"\nStatistics:")
        print(f"  Total characters: {stats['total_characters']}")
        print(f"  Estimated tokens: {stats['estimated_tokens']}")
        print(f"  Pages with content: {stats['pages_with_content']}")
        
        # Extract first page
        first_page_text = extractor.get_page_text(1)
        print(f"\nFirst page preview (first 200 chars):")
        print(first_page_text[:200] + "...")

=== Testing PDF Extraction ===
PDF: ARN44767-FM_3-01-000-WEB-1.pdf
Pages: 218

Statistics:
  Total characters: 684001
  Estimated tokens: 171000
  Pages with content: 217

First page preview (first 200 chars):
FM 3-01
U.S. Army Air and Missile
Defense Operations
AUGUST 2025
DISTRIBUTION RESTRICTION:
Approved for public release; distribution is unlimited.
This publication supersedes FM 3-01, dated 22 Decembe...


In [ ]:
import re
import numpy as np
from typing import List, Dict, Tuple
from dataclasses import dataclass

@dataclass
class SemanticChunk:
    """Data class for a semantic chunk"""
    id: str
    text: str
    sentences: List[str]
    page_numbers: List[int]
    start_position: int
    end_position: int
    token_count: int
    embedding: List[float]
    quality_score: float
    metadata: Dict

class SemanticChunker:
    """
    Semantic chunker using Qwen3-Embedding-4B
    """
    
    def __init__(self, embedding_model: QwenEmbeddingModel, 
                 similarity_threshold: float = 0.75,
                 min_chunk_length: int = 5):
        """
        Initialize semantic chunker
        
        Args:
            embedding_model: QwenEmbeddingModel instance
            similarity_threshold: Threshold for semantic similarity (0-1)
                                 Lower = split more often
                                 Higher = combine more chunks
            min_chunk_length: Minimum characters per chunk
        """
        self.embedding_model = embedding_model
        self.similarity_threshold = similarity_threshold
        self.min_chunk_length = min_chunk_length
    
    def split_into_sentences(self, text: str) -> List[Tuple[str, int]]:
        """
        Split text into sentences
        
        Returns:
            List of (sentence, character_position) tuples
        """
        # Enhanced sentence splitting
        # Handles: ".", "!", "?", abbreviations, etc.
        
        sentences = []
        position = 0
        
        # Split on sentence boundaries
        sentence_pattern = r'(?<=[.!?])\s+(?=[A-Z])'
        splits = re.split(sentence_pattern, text)
        
        for sentence in splits:
            sentence = sentence.strip()
            if sentence:
                sentences.append((sentence, position))
                position += len(sentence)
        
        return sentences
    
    def _calculate_similarity(self, embedding1: List[float], 
                             embedding2: List[float]) -> float:
        """
        Calculate cosine similarity between two embeddings
        
        Args:
            embedding1: First embedding vector
            embedding2: Second embedding vector
        
        Returns:
            Similarity score (0-1)
        """
        a = np.array(embedding1)
        b = np.array(embedding2)
        
        # Cosine similarity
        similarity = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
        return float(similarity)
    
    def chunk_document(self, text: str, document_id: str = "doc_1",
                      source_file: str = None) -> List[SemanticChunk]:
        """
        Chunk document using semantic boundaries
        
        Args:
            text: Document text
            document_id: Identifier for document
            source_file: Original file path
        
        Returns:
            List of SemanticChunk objects
        """
        print("\n=== SEMANTIC CHUNKING ===")
        
        # Step 1: Split into sentences
        print("Step 1: Splitting into sentences...")
        sentences = self.split_into_sentences(text)
        print(f"  Created {len(sentences)} sentences")
        
        if len(sentences) < 2:
            # Document is too short, return as single chunk
            chunk = SemanticChunk(
                id=f"{document_id}_0",
                text=text,
                sentences=[s[0] for s in sentences],
                page_numbers=[1],
                start_position=0,
                end_position=len(text),
                token_count=len(text) // 4,
                embedding=self.embedding_model.embed_single(text),
                quality_score=0.8,
                metadata={"source_file": source_file}
            )
            return [chunk]
        
        # Step 2: Generate embeddings for each sentence
        print("Step 2: Generating embeddings...")
        sentence_texts = [s[0] for s in sentences]
        sentence_embeddings = self.embedding_model.embed_texts(sentence_texts)
        print(f"  Generated {len(sentence_embeddings)} embeddings")
        
        # Step 3: Calculate similarities between consecutive sentences
        print("Step 3: Calculating semantic similarities...")
        similarities = []
        for i in range(len(sentence_embeddings) - 1):
            sim = self._calculate_similarity(
                sentence_embeddings[i],
                sentence_embeddings[i + 1]
            )
            similarities.append(sim)
        
        # Step 4: Find split points (where similarity drops below threshold)
        print(f"Step 4: Finding split points (threshold: {self.similarity_threshold})...")
        split_indices = [0]  # Always start with first sentence
        
        for i, sim in enumerate(similarities):
            if sim < self.similarity_threshold:
                split_indices.append(i + 1)  # Split after sentence i
        
        split_indices.append(len(sentences))  # Always end with last sentence
        
        # Remove duplicates and sort
        split_indices = sorted(set(split_indices))
        print(f"  Found {len(split_indices) - 1} potential chunks")
        
        # Step 5: Create chunks
        print("Step 5: Creating chunks...")
        chunks = []
        chunk_id = 0
        
        for i in range(len(split_indices) - 1):
            start_idx = split_indices[i]
            end_idx = split_indices[i + 1]
            
            # Get sentences for this chunk
            chunk_sentences = sentence_texts[start_idx:end_idx]
            chunk_text = " ".join(chunk_sentences)
            
            # Skip very small chunks
            if len(chunk_text) < self.min_chunk_length:
                continue
            
            # Get position info
            start_pos = sentences[start_idx][1]
            end_pos = sentences[end_idx - 1][1] + len(sentence_texts[end_idx - 1])
            
            # Average embedding for chunk
            chunk_embedding = np.mean(
                sentence_embeddings[start_idx:end_idx],
                axis=0
            ).tolist()
            
            # Calculate quality score
            quality = self._calculate_chunk_quality(chunk_text)
            
            chunk = SemanticChunk(
                id=f"{document_id}_chunk_{chunk_id}",
                text=chunk_text,
                sentences=chunk_sentences,
                page_numbers=[1],  # Will be updated if we have page info
                start_position=start_pos,
                end_position=end_pos,
                token_count=len(chunk_text) // 4,
                embedding=chunk_embedding,
                quality_score=quality,
                metadata={
                    "source_file": source_file,
                    "chunk_index": chunk_id,
                    "sentence_count": len(chunk_sentences),
                    "similarity_threshold": self.similarity_threshold
                }
            )
            
            chunks.append(chunk)
            chunk_id += 1
        
        print(f"Step 5 complete: Created {len(chunks)} final chunks")
        return chunks
    
    def _calculate_chunk_quality(self, text: str) -> float:
        """
        Calculate quality score for chunk (0-1)
        
        Factors:
        - Length (optimal: 200-500 chars)
        - Completeness (starts with capital, ends with punctuation)
        - Sentence count (minimum 2)
        """
        score = 0.0
        
        # Length check
        if 200 < len(text) < 500:
            score += 0.4
        elif len(text) > 100:
            score += 0.2
        
        # Completeness
        if text[0].isupper() and text[-1] in '.!?':
            score += 0.3
        elif text[0].isupper() or text[-1] in '.!?':
            score += 0.15
        
        # Sentence count
        sentences = text.split('. ')
        if len(sentences) >= 2:
            score += 0.3
        elif len(sentences) >= 1:
            score += 0.15
        
        return min(score, 1.0)

# Test the semantic chunker
if __name__ == "__main__":
    print("=== Testing Semantic Chunking ===\n")
    
    # Initialize embedding model
    print("Loading Qwen embedding model...")
    embedding_model = QwenEmbeddingModel()
    
    # Initialize chunker
    chunker = SemanticChunker(
        embedding_model=embedding_model,
        similarity_threshold=0.75,
        min_chunk_length=100
    )
    
    # Sample text
    sample_text = """
    Personal information is any data that identifies an individual. This includes names, 
    addresses, phone numbers, and identification numbers. Organizations must handle personal 
    information carefully. Data collection must follow specific procedures. Consent is required 
    before collecting sensitive data. Storage requires encryption and security measures. 
    Access to personal information should be restricted. Data breaches must be reported immediately. 
    Individuals have the right to access their data. Corrections can be requested anytime.
    """
    
    # Chunk the document
    chunks = chunker.chunk_document(sample_text, document_id="test_doc")
    
    # Display results
    print("\n=== RESULTS ===")
    print(f"Total chunks: {len(chunks)}")
    print(f"Average quality: {np.mean([c.quality_score for c in chunks]):.2f}\n")
    
    for i, chunk in enumerate(chunks):
        print(f"Chunk {i}:")
        print(f"  Length: {len(chunk.text)} chars")
        print(f"  Quality: {chunk.quality_score:.2f}")
        print(f"  Text: {chunk.text[:100]}...\n")

=== Testing Semantic Chunking ===

Loading Qwen embedding model...
Using device: cpu
Loading Qwen/Qwen3-Embedding-4B...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]